# 📝 LangChain 기본 구조 과제 LV1 정답 — 모델·프롬프트·체인 (강사용)

각 문제의 **모범답안 + 해설**입니다. 경로는 `../../day18_LangChain_기본구조/data/` 입니다.

- 순수 함수·구조 문제는 결정적으로 채점합니다.
- 모델 출력은 **타입·구조**(문자열인지, 리스트 길이 등)로만 채점합니다 — 실제 모델을 부르므로 문장은 학생마다·실행마다 다릅니다.

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../../day18_LangChain_기본구조/.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이 과제에서 공통으로 쓰는 부품들 — 실행하면 준비 끝입니다.
#   AIMessage 는 자가채점이 '응답이 정말 모델 응답 부품인지' 확인할 때 씁니다.
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
parser = StrOutputParser()
print('부품 준비 완료 —', type(model).__name__, '+', type(parser).__name__)

## 1. 모델에게 물어보기
**배경**: 준비 셀이 만든 `model` 로 질문을 하나 던집니다.

**요구사항**:
- `model.invoke(...)` 로 질문 **"중고 전자기기를 안전하게 거래하려면 무엇을 확인해야 할까? 두 가지만 짧게 알려줘."** 을 던진 결과(응답 부품)를 변수 **`reply1`** 에 담으세요.
- 응답의 순수 텍스트를 변수 **`text1`** 에 담으세요(`.text` 사용).

**예시**: `reply1` 은 `AIMessage`, `text1` 은 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- model.invoke 에 질문 문자열을 넣고, 응답에서 .text 로 텍스트를 꺼낸다.

세부구현:
1. model.invoke(질문) 결과를 reply1 에 담는다.
2. reply1.text 를 text1 에 담는다.
```

</details>

In [ ]:
# 문자열 하나만 넣으면 사용자의 말(HumanMessage)로 취급된다 — 역할이 필요 없을 때 가장 짧은 형태.
reply1 = model.invoke('중고 전자기기를 안전하게 거래하려면 무엇을 확인해야 할까? 두 가지만 짧게 알려줘.')
# .text 는 응답이 어떤 모양이든 사람이 읽을 텍스트만 돌려준다.
text1 = reply1.text
print(text1)

In [ ]:
# [자가채점] 모델이 만든 문장은 매번 다르므로, 여기서는 '무엇이 담겼는지'만 확인합니다.
assert isinstance(reply1, AIMessage)          # invoke 의 결과는 응답 부품(AIMessage)
assert reply1.response_metadata               # 실제 호출로 받은 응답에만 모델·토큰 정보가 붙어 있다
assert isinstance(text1, str) and len(text1.strip()) > 0
assert text1 == reply1.text                   # .text 로 꺼낸 것인지
print('✅ 통과!')

**해설**: `model.invoke(문자열)` 은 문자열을 사용자의 말로 취급합니다. 응답은 `AIMessage` 부품이고, 사람이 읽을 텍스트는 `.text` 로 꺼냅니다(`.content` 는 조각 리스트일 수 있어 `.text` 가 안전).

## 2. 역할을 정해 물어보기
**배경**: `SystemMessage` 로 모델의 역할을 정하면 답의 태도가 달라집니다.

**요구사항**:
- **`SystemMessage`** 로 역할을 `'너는 중고 거래 플랫폼의 친절한 안전거래 안내원이다. 존댓말로 간결하게 답한다.'` 로 주고, **`HumanMessage`** 로 질문 **"직거래 장소는 어디가 좋을까?"** 을 담은 **리스트**를 변수 **`messages2`** 에 담으세요(순서는 역할이 먼저).
- 그 리스트를 `model.invoke(...)` 에 넘긴 결과의 텍스트를 변수 **`text2`** 에 담으세요.

**예시**: `messages2` 는 원소 두 개짜리 리스트이고, `text2` 는 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- SystemMessage 와 HumanMessage 를 리스트로 묶어 invoke 하고 .text 를 꺼낸다.

세부구현:
1. [SystemMessage(역할), HumanMessage(질문)] 리스트를 messages2 에 담는다.
2. model.invoke(messages2).text 를 text2 에 담는다.
```

</details>

In [ ]:
# 메시지가 둘 이상이면 리스트로 넣는다. 역할(System)이 앞, 사용자의 말(Human)이 뒤.
messages2 = [
    SystemMessage('너는 중고 거래 플랫폼의 친절한 안전거래 안내원이다. 존댓말로 간결하게 답한다.'),
    HumanMessage('직거래 장소는 어디가 좋을까?'),
]
text2 = model.invoke(messages2).text
print(text2)

In [ ]:
# [자가채점] 답의 '내용'은 채점할 수 없으니, 역할·질문 메시지를 제대로 만들었는지를 봅니다.
assert isinstance(messages2, list) and len(messages2) == 2
assert isinstance(messages2[0], SystemMessage) and isinstance(messages2[1], HumanMessage)
assert isinstance(text2, str) and len(text2.strip()) > 0
print('✅ 통과!')

**해설**: 여러 메시지는 **리스트**로 넣습니다. `SystemMessage` 는 역할·태도를, `HumanMessage` 는 사용자 질문을 담습니다. 역할을 바꾸면 같은 질문이라도 답의 결이 달라집니다.

## 3. 한 줄 템플릿 — `from_template`
**배경**: 역할(system)을 따로 줄 필요 없이 **사람 메시지 하나**만 필요할 때는 `ChatPromptTemplate.from_template(...)` 로 더 짧게 만들 수 있습니다(교안 3절 보조 표기). 채우는 방법은 `from_messages` 로 만든 것과 똑같습니다.

**요구사항**:
- `ChatPromptTemplate.from_template('{name} 를 중고로 살 때 꼭 확인할 점을 한 문장으로 알려줘.')` 로 프롬프트 **`quick_prompt`** 를 만드세요.
- `quick_prompt.invoke({'name': '무선 이어폰'})` 로 값을 채운 뒤 `model` 에 넣어 응답 텍스트를 변수 **`text3`** 에 담으세요(`.text`).

**예시**: `quick_prompt` 의 변수는 `name` 하나이고, `text3` 는 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- from_template 은 사람 메시지 한 줄짜리 템플릿을 만든다. 채우는 방법(invoke)은 from_messages 와 같다.

세부구현:
1. ChatPromptTemplate.from_template(문자열) 로 quick_prompt 를 만든다.
2. quick_prompt.invoke({'name': 값}) 로 프롬프트를 완성한다.
3. 완성된 프롬프트를 model.invoke 에 넣고 .text 를 text3 에 담는다.
```

</details>

In [ ]:
# from_template 은 사람 메시지 한 줄짜리 — 역할(system)을 나눌 필요가 없을 때만 쓴다.
quick_prompt = ChatPromptTemplate.from_template('{name} 를 중고로 살 때 꼭 확인할 점을 한 문장으로 알려줘.')
filled3 = quick_prompt.invoke({'name': '무선 이어폰'})
text3 = model.invoke(filled3).text
print(text3)

In [ ]:
# [자가채점] 템플릿을 진짜로 만들었는지(변수 이름까지)와 답이 왔는지를 함께 봅니다.
assert set(quick_prompt.input_variables) == {'name'}   # 문자열만 넘기고 끝냈다면 여기서 걸린다
assert isinstance(text3, str) and len(text3.strip()) > 0
print('✅ 통과!')

**해설**: `from_template` 은 **사람 메시지 하나**짜리 템플릿을 만드는 짧은 표기입니다. 역할(system)을 나눠야 하면 4·5번처럼 `from_messages` 를 씁니다. 어느 쪽으로 만들든 채우는 방법(`invoke({'변수': 값})`)과 이어 붙이는 방법(파이프 `|`)은 똑같습니다.

## 4. 프롬프트 템플릿 — 변수 하나
**배경**: 같은 형식의 요청을 값만 바꿔 재사용하려고 프롬프트 템플릿을 만듭니다.

**요구사항**:
- `ChatPromptTemplate.from_messages` 로 `('human', '{name} 를 중고로 팔 때 쓸 짧은 소개 문구를 한 문장으로 써줘.')` 프롬프트 **`intro_prompt`** 를 만드세요.
- `intro_prompt.invoke({'name': '무선 이어폰'})` 로 값을 채운 뒤 `model` 에 넣어 응답 텍스트를 변수 **`intro1`** 에 담으세요.

**예시**: `intro1` 은 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 템플릿을 만들고 invoke 로 값을 채운 뒤 모델에 넣고 .text 를 꺼낸다.

세부구현:
1. ChatPromptTemplate.from_messages 로 {name} 을 받는 프롬프트를 만든다.
2. intro_prompt.invoke({'name': 값}) 로 프롬프트를 완성한다.
3. 완성된 프롬프트를 model.invoke 에 넣고 .text 를 intro1 에 담는다.
```

</details>

In [ ]:
intro_prompt = ChatPromptTemplate.from_messages([
    ('human', '{name} 를 중고로 팔 때 쓸 짧은 소개 문구를 한 문장으로 써줘.'),
])
# 템플릿은 만들 때가 아니라 invoke 로 값을 채울 때 비로소 완성된 프롬프트가 된다.
filled = intro_prompt.invoke({'name': '무선 이어폰'})
intro1 = model.invoke(filled).text
print(intro1)

In [ ]:
# [자가채점] 템플릿을 진짜로 만들었는지(변수 이름까지)와 답이 왔는지를 함께 봅니다.
assert set(intro_prompt.input_variables) == {'name'}   # 손으로 문장을 적고 끝냈다면 여기서 걸린다
assert isinstance(intro1, str) and len(intro1.strip()) > 0
print('✅ 통과!')

**해설**: 템플릿의 `{name}` 자리는 `invoke({'name': 값})` 로 채웁니다. 값만 바꾸면 같은 형식의 문구를 얼마든지 만들 수 있습니다.

## 5. 프롬프트 템플릿 — 변수 두 개
**배경**: 변수가 여러 개인 템플릿도 같은 방식입니다(4번과 다른 유형 — 변수 2개).

**요구사항**:
- `('human', '{name} (상태 {condition}) 를 소개하는 문구를 한 문장으로 써줘.')` 프롬프트 **`grade_prompt`** 를 만드세요.
- `grade_prompt.invoke({'name': '게이밍 노트북', 'condition': '중'})` 로 채워 `model` 에 넣고 응답 텍스트를 변수 **`intro2`** 에 담으세요.

**예시**: `intro2` 는 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4번과 같되 변수를 name·condition 두 개로 둔다.

세부구현:
1. {name}·{condition} 두 변수를 가진 프롬프트를 만든다.
2. invoke 에 두 값을 딕셔너리로 넘겨 채운다.
3. 모델에 넣고 .text 를 intro2 에 담는다.
```

</details>

In [ ]:
# 변수가 늘어도 방법은 같다 — invoke 딕셔너리에 키를 그만큼 채워 주면 된다.
grade_prompt = ChatPromptTemplate.from_messages([
    ('human', '{name} (상태 {condition}) 를 소개하는 문구를 한 문장으로 써줘.'),
])
filled2 = grade_prompt.invoke({'name': '게이밍 노트북', 'condition': '중'})
intro2 = model.invoke(filled2).text
print(intro2)

In [ ]:
# [자가채점] 변수가 둘이어도 결과 형태는 같습니다 — 다만 변수는 둘 다 있어야 합니다.
assert set(grade_prompt.input_variables) == {'name', 'condition'}
assert isinstance(intro2, str) and len(intro2.strip()) > 0
print('✅ 통과!')

**해설**: 변수가 여러 개면 `invoke` 에 그만큼 키를 채워 넘깁니다. 프롬프트 형식은 하나로 두고 값만 바꿉니다.

## 6. 출력 파서로 문자열 뽑기
**배경**: 준비 셀의 `parser`(StrOutputParser)는 모델 응답(`AIMessage`)을 문자열로 바꿔 줍니다. 아래 **제공 셀**이 만들어 둔 응답 `given_reply` 를 파서에 통과시켜 봅니다.

In [ ]:
# [제공 코드] 채점용 응답 하나를 미리 만들어 둡니다.
given_reply = model.invoke('중고 노트북을 살 때 배터리 상태는 어떻게 확인해?')

**요구사항**:
- `parser.invoke(given_reply)` 결과를 변수 **`parsed6`** 에 담으세요.

**예시**: `parsed6` 은 문자열(`str`)입니다.

<details><summary>힌트</summary>

```text
접근방법:
- parser 의 invoke 에 AIMessage 를 넣으면 문자열이 나온다.

세부구현:
1. parser.invoke(given_reply) 를 parsed6 에 담는다.
```

</details>

In [ ]:
# 파서는 AIMessage 를 받아 문자열을 낸다 — 이 부품을 체인 끝에 두면 .text 를 부를 일이 없어진다(7번).
parsed6 = parser.invoke(given_reply)
print(type(parsed6).__name__)
print(parsed6)

In [ ]:
# [자가채점] 파서를 통과했으면 AIMessage 가 아니라 str 이어야 합니다.
assert isinstance(parsed6, str) and len(parsed6.strip()) > 0
print('✅ 통과!')

**해설**: `StrOutputParser` 는 `AIMessage` 를 문자열로 바꿔 줍니다. 이 파서를 체인 끝에 두면 매번 `.text` 를 부르지 않아도 됩니다(다음 문제).

## 7. 체인 조립 — 프롬프트 | 모델 | 파서
**배경**: 프롬프트·모델·파서를 파이프 `|` 로 이어 하나의 체인으로 만듭니다.

**요구사항**:
- 4번의 `intro_prompt` 에 `model`, `parser` 를 파이프로 이어 체인 **`intro_chain`** 을 만드세요.
- `intro_chain.invoke({'name': '무선 이어폰'})` 결과를 변수 **`out7`** 에 담으세요.

**예시**: `out7` 은 문자열입니다(파서가 끝에 있어 바로 문자열).

<details><summary>힌트</summary>

```text
접근방법:
- 프롬프트 | 모델 | 파서 를 파이프로 잇는다.

세부구현:
1. intro_prompt | model | parser 로 intro_chain 을 만든다.
2. intro_chain.invoke({'name': 값}) 를 out7 에 담는다.
```

</details>

In [ ]:
# 파이프는 왼쪽 출력을 오른쪽 입력으로 넘긴다. 파서가 끝에 있으니 결과가 AIMessage 가 아니라 바로 str 이다.
intro_chain = intro_prompt | model | parser
out7 = intro_chain.invoke({'name': '무선 이어폰'})
print(type(out7).__name__)
print(out7)

In [ ]:
# [자가채점] 체인의 부품 순서를 직접 들여다봅니다 — steps 는 파이프로 이은 부품들의 목록입니다.
assert isinstance(intro_chain.steps[-1], StrOutputParser)   # 파서가 맨 끝에 있어야 결과가 str 이다
assert isinstance(intro_chain.steps[0], ChatPromptTemplate)
assert isinstance(out7, str) and len(out7.strip()) > 0
print('✅ 통과!')

**해설**: 파이프 `|` 는 왼쪽 출력을 오른쪽 입력으로 넘깁니다. 파서가 끝에 있으니 `invoke` 결과가 바로 문자열입니다 — `.text` 를 따로 부를 필요가 없습니다.

## 8. 체인 조립 — 다른 프롬프트로
**배경**: 프롬프트만 바꿔 다른 일을 하는 체인을 만듭니다(7번과 다른 조합).

**요구사항**:
- `('human', '{name} 의 상태({condition})를 중고 구매자에게 솔직하게 설명하는 한 문장을 써줘.')` 프롬프트를 만들고 `model`, `parser` 를 이어 체인 **`explain_chain`** 을 만드세요.
- `explain_chain.invoke({'name': '게이밍 노트북', 'condition': '중'})` 결과를 변수 **`out8`** 에 담으세요.

**예시**: `out8` 은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 7번과 같되 프롬프트 내용·변수를 바꾼다.

세부구현:
1. {name}·{condition} 을 받는 설명 프롬프트를 만든다.
2. 프롬프트 | model | parser 로 explain_chain 을 만든다.
3. invoke 에 두 값을 넘겨 out8 에 담는다.
```

</details>

In [ ]:
explain_prompt = ChatPromptTemplate.from_messages([
    ('human', '{name} 의 상태({condition})를 중고 구매자에게 솔직하게 설명하는 한 문장을 써줘.'),
])
# 체인 구조(프롬프트|모델|파서)는 그대로 두고 프롬프트만 갈아 끼웠다 — 재사용의 핵심.
explain_chain = explain_prompt | model | parser
out8 = explain_chain.invoke({'name': '게이밍 노트북', 'condition': '중'})
print(out8)

In [ ]:
# [자가채점] 7번과 같은 형태 — 프롬프트만 바뀌었을 뿐입니다.
assert isinstance(explain_chain.steps[-1], StrOutputParser)
assert set(explain_chain.steps[0].input_variables) == {'name', 'condition'}
assert isinstance(out8, str) and len(out8.strip()) > 0
print('✅ 통과!')

**해설**: 체인 구조는 그대로 두고 **프롬프트만 바꾸면** 다른 일을 하는 체인이 됩니다 — 재사용의 핵심입니다.

## 9. 여러 입력을 한꺼번에 — batch
**배경**: 상품 여러 개의 문구를 한 번에 만들 때 `batch` 를 씁니다.

**요구사항**:
- 아래 세 입력을 리스트로 만들어 변수 **`inputs9`** 에 담으세요.
- 입력: `[{'name': '무선 이어폰'}, {'name': '태블릿 10인치'}, {'name': '블루투스 스피커'}]`
- 7번의 `intro_chain` 에 그 리스트를 `batch` 로 넘긴 결과 리스트를 변수 **`results9`** 에 담으세요.

**예시**: `results9` 는 길이 3의 리스트이고, 각 원소는 문자열입니다(상품이 다르니 문구도 서로 다릅니다).

<details><summary>힌트</summary>

```text
접근방법:
- 입력 딕셔너리들의 리스트를 batch 에 넘긴다.

세부구현:
1. 세 입력을 리스트로 만들어 inputs9 에 담는다.
2. intro_chain.batch(입력리스트) 를 results9 에 담는다.
```

</details>

In [ ]:
inputs9 = [{'name': '무선 이어폰'}, {'name': '태블릿 10인치'}, {'name': '블루투스 스피커'}]
# batch 는 세 요청을 동시에 보내지만 결과는 입력과 같은 순서로 돌려준다(그래서 zip 으로 짝지어도 안전).
results9 = intro_chain.batch(inputs9)
for r in results9:
    print('-', r)

In [ ]:
# [자가채점] batch 는 입력 개수만큼의 결과 리스트를 돌려줍니다.
assert inputs9 == [{'name': '무선 이어폰'}, {'name': '태블릿 10인치'}, {'name': '블루투스 스피커'}]
#   batch 가 받는 입력은 '딕셔너리들의 리스트' — 문자열 리스트를 넘기면 프롬프트 변수를 못 채운다
assert isinstance(results9, list) and len(results9) == 3
assert all(isinstance(r, str) and len(r.strip()) > 10 for r in results9)
assert len(set(results9)) == 3   # 상품이 셋이니 문구도 서로 달라야 한다
print('✅ 통과!')

**해설**: `batch` 는 입력 리스트를 받아 **결과 리스트**를 돌려줍니다. 세 요청을 동시에 보내 빠르고, 결과는 **입력 순서 그대로** 돌아옵니다. 요금제의 분당 요청 한도가 걱정되면 `config={'max_concurrency': 1}` 로 한 번에 하나씩 보낼 수 있습니다.

## 10. 답을 조각으로 흘려 받기 — stream
**배경**: 긴 답을 기다리지 않고 조각(청크) 단위로 받아 이어 붙입니다.

**요구사항**:
- 7번의 `intro_chain` 을 `stream` 으로 실행해 나오는 **조각들을 리스트** `pieces10` 에 모으세요.
- 입력은 `{'name': '무선 이어폰'}` 을 쓰세요.
- 조각들을 이어 붙인 전체 문자열을 변수 **`full10`** 에 담으세요(`''.join(...)`).

**예시**: `pieces10` 은 리스트, `full10` 은 비어 있지 않은 문자열입니다. (조각 개수는 실행 환경마다 다를 수 있으니 **개수로 채점하지 않습니다** — 이어 붙인 문자열로 확인합니다.)

<details><summary>힌트</summary>

```text
접근방법:
- for 문으로 stream 의 조각을 리스트에 모으고, join 으로 잇는다.

세부구현:
1. 빈 리스트 pieces10 을 만든다.
2. for chunk in intro_chain.stream(입력): 으로 조각을 append 한다.
3. ''.join(pieces10) 을 full10 에 담는다.
```

</details>

In [ ]:
# 조각 개수는 모델이 정하므로 실행마다 다르다 — 개수가 아니라 이어 붙인 문자열로 다룬다.
pieces10 = []
for chunk in intro_chain.stream({'name': '무선 이어폰'}):
    pieces10.append(chunk)
full10 = ''.join(pieces10)
print('조각 개수:', len(pieces10))
print('전체 답:', full10)

In [ ]:
# [자가채점] 조각 '개수' 는 실행마다 달라 정확한 수로 채점하지 않습니다.
assert isinstance(pieces10, list) and len(pieces10) > 1   # 스트리밍이면 조각이 여러 개다
assert all(isinstance(c, str) for c in pieces10)
assert isinstance(full10, str) and len(full10.strip()) > 0
assert full10 == ''.join(pieces10)   # 모은 조각과 이어 붙인 문자열이 서로 맞는지
print('✅ 통과!')

**해설**: `stream` 은 답을 **조각**으로 흘려 줍니다. 조각을 나누는 기준은 모델이 정하므로 **개수는 실행할 때마다 다릅니다** — 그래서 조각을 **이어 붙인 문자열**로 다룹니다.

## 11. 사진을 보고 매물 소개 문구 쓰기
**배경**: 메시지에는 글자뿐 아니라 **사진**도 담을 수 있습니다(교안 2절). 중고 매물 사진을 모델에게 보여 주고 소개 문구를 받아 봅니다.

**요구사항**:
- 아래 **제공 셀**이 만든 `photo_url`(`../../day18_LangChain_기본구조/data/images/used_earbuds.jpg` 의 data URL)을 그대로 쓰세요.
- `HumanMessage` 의 **`content` 를 블록 리스트**로 만드세요 — 글자 블록 하나와 사진 블록 하나.
  - 글자 블록의 질문: **"이 중고 매물 사진을 보고 판매글에 쓸 소개 문구를 두 문장으로 써줘. 보이는 상태(사용감 등)를 솔직하게 포함해줘."**
- 그 메시지 하나를 리스트에 담아 `model.invoke(...)` 한 결과를 변수 **`reply11`** 에, 응답 텍스트를 변수 **`text11`** 에 담으세요.

**예시**: `reply11` 은 `AIMessage`, `text11` 은 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 2절의 이미지 예시와 같은 모양이다. 사진 블록의 url 자리에 photo_url 을 넣는다.

세부구현:
1. content 리스트에 글자 블록과 사진 블록을 차례로 담는다.
2. HumanMessage(content=그 리스트) 하나를 리스트에 담아 model.invoke 에 넘긴다.
3. 결과를 reply11 에, 그 .text 를 text11 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 매물 사진을 data URL 로 바꿔 둡니다 — 교안 2절의 to_data_url 과 같은 방법입니다.
import base64

with open('../../day18_LangChain_기본구조/data/images/used_earbuds.jpg', 'rb') as f:
    photo_url = 'data:image/jpeg;base64,' + base64.b64encode(f.read()).decode()
print('사진 준비 완료 / data URL 길이:', len(photo_url), '자')

모델에게 보여 줄 매물 사진입니다.

<img src="../../day18_LangChain_기본구조/data/images/used_earbuds.jpg" width="260">

In [ ]:
# 글자 블록과 사진 블록을 함께 담는다 — 메시지 종류(HumanMessage)와 .text 는 글자만 보낼 때와 같다.
content11 = [
    {'type': 'text', 'text': '이 중고 매물 사진을 보고 판매글에 쓸 소개 문구를 두 문장으로 써줘. 보이는 상태(사용감 등)를 솔직하게 포함해줘.'},
    {'type': 'image_url', 'image_url': {'url': photo_url}},
]
reply11 = model.invoke([HumanMessage(content=content11)])
text11 = reply11.text
print(text11)

In [ ]:
# [자가채점] 문장 내용은 채점할 수 없으니, 사진 블록을 제대로 실어 보냈는지를 봅니다.
assert isinstance(content11, list) and len(content11) == 2
kinds = {b['type'] for b in content11}
assert kinds == {'text', 'image_url'}                 # 글자 블록과 사진 블록이 하나씩
assert content11[1]['image_url']['url'] == photo_url  # 제공된 사진을 그대로 넣었는지
assert isinstance(reply11, AIMessage) and reply11.response_metadata
assert isinstance(text11, str) and len(text11.strip()) > 0
print('✅ 통과!')

**해설**: 사진을 보내려면 `HumanMessage` 의 **`content` 를 블록 리스트**로 줍니다 — 글자 블록(`{'type': 'text', ...}`)과 사진 블록(`{'type': 'image_url', ...}`). 메시지 종류도, 답을 꺼내는 `.text` 도 글자만 보낼 때와 **똑같습니다**. 바뀌는 것은 **내용의 모양** 하나뿐입니다.

> 채점이 문장이 아니라 **블록 구성**을 보는 이유: 모델이 쓴 소개 문구는 실행마다 달라 정확히 맞출 수 없지만, **사진을 제대로 실어 보냈는지**는 결정적으로 확인할 수 있기 때문입니다. 그리고 모델은 사진에 없는 것도 지어내므로, 답은 항상 사진과 대조해 보세요.

## 12. 프롬프트를 파일로 만들어 읽어 쓰기
**배경**: 프롬프트를 코드에 박지 않고 **파일**에 두면 코드를 고치지 않고 문구만 바꿀 수 있습니다(교안 3절). 이번엔 그 파일을 **직접 만들어** 봅니다.

**요구사항**:
- **`output/my_prompt.yml`** 파일을 직접 만드세요. 폴더가 없으면 먼저 만들어야 합니다.
- 파일 내용은 아래와 **똑같은 구조**여야 합니다(문구는 바꿔도 됩니다). `human` 에는 반드시 **`{name}` 과 `{condition}` 두 변수**가 들어가야 합니다.

```yaml
seller_intro:
  description: 매물 이름과 상태를 받아 판매글 소개 문구를 쓴다
  system: |
    너는 중고 거래 판매글을 다듬어 주는 편집자다.
  human: |
    다음 매물의 소개 문구를 한 문장으로 써줘.
    이름: {name}
    상태: {condition}
```

- 만든 파일을 **다시 읽어**(`yaml.safe_load`) `seller_intro` 항목을 변수 **`spec12`** 에 담으세요.
- 그 `system`·`human` 으로 `ChatPromptTemplate.from_messages` 프롬프트 **`file_prompt12`** 를 만드세요.
- `{'name': '태블릿 10인치', 'condition': '상'}` 을 채워 `model` 에 넣고 응답 텍스트를 변수 **`text12`** 에 담으세요.

**예시**: 실행 후 `output/my_prompt.yml` 이 생기고, `file_prompt12` 가 요구하는 변수는 `name`·`condition` 두 개입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 파일을 쓰는 것도 읽는 것도 4일차에 배운 open 이다. YAML 변환만 yaml 모듈이 해 준다.

세부구현:
1. os 모듈로 저장할 폴더를 미리 만들어 둔다(이미 있어도 오류가 나지 않게).
2. 위 구조와 같은 YAML 글을 문자열로 만들어 파일에 쓴다(들여쓰기가 구조를 만드니 그대로 지킨다).
3. 그 파일을 다시 열어 yaml.safe_load 로 딕셔너리를 얻고, seller_intro 항목을 spec12 에 담는다.
4. spec12 의 system·human 으로 프롬프트를 만들고, 두 변수를 채워 모델에 넣는다.
```

</details>

In [ ]:
import os

import yaml

# 1) 저장할 폴더를 먼저 준비한다(exist_ok=True 라 이미 있어도 오류가 나지 않는다).
os.makedirs('output', exist_ok=True)

# 2) YAML 은 '들여쓰기'가 구조다 — 아래 모양 그대로 써야 seller_intro 아래에 세 항목이 들린다.
#    human 의 여러 줄은 | 뒤에 그대로 쓴다(JSON 처럼 줄바꿈을 \n 으로 바꿔 적을 필요가 없다).
yml_text = '''seller_intro:
  description: 매물 이름과 상태를 받아 판매글 소개 문구를 쓴다
  system: |
    너는 중고 거래 판매글을 다듬어 주는 편집자다.
  human: |
    다음 매물의 소개 문구를 한 문장으로 써줘.
    이름: {name}
    상태: {condition}
'''
with open('output/my_prompt.yml', 'w', encoding='utf-8') as f:
    f.write(yml_text)

# 3) 방금 쓴 파일을 '다시 읽어' 쓴다 — 실제로 파일에서 오는지 확인하는 셈이다.
with open('output/my_prompt.yml', encoding='utf-8') as f:
    spec12 = yaml.safe_load(f)['seller_intro']

# 4) 파일에서 온 글이든 코드에 적은 글이든 from_messages 입장에서는 똑같은 문자열이다.
file_prompt12 = ChatPromptTemplate.from_messages([
    ('system', spec12['system']),
    ('human', spec12['human']),
])
filled12 = file_prompt12.invoke({'name': '태블릿 10인치', 'condition': '상'})
text12 = model.invoke(filled12).text
print('요구 변수:', sorted(file_prompt12.input_variables))
print(text12)

In [ ]:
# [자가채점] 파일이 진짜로 만들어졌는지, 그리고 그 파일에서 읽어 만들었는지를 봅니다.
import os

import yaml

assert os.path.exists('output/my_prompt.yml')            # 파일을 실제로 만들었는지
with open('output/my_prompt.yml', encoding='utf-8') as f:
    saved = yaml.safe_load(f)
assert 'seller_intro' in saved                         # 항목 이름이 맞는지
assert set(saved['seller_intro']) >= {'system', 'human'}
assert '{name}' in saved['seller_intro']['human'] and '{condition}' in saved['seller_intro']['human']
assert spec12 == saved['seller_intro']                 # 코드에 적지 않고 파일에서 읽어 왔는지
assert set(file_prompt12.input_variables) == {'name', 'condition'}
assert isinstance(text12, str) and len(text12.strip()) > 0
print('✅ 통과!')

**해설**: 프롬프트를 파일로 빼면 **코드를 고치지 않고 문구만** 바꿀 수 있습니다. YAML 을 쓰는 이유는 프롬프트가 여러 줄이기 때문입니다 — `|` 뒤에 쓴 글이 **파일에 보이는 그대로** 프롬프트가 되어, JSON 처럼 줄바꿈을 `\n` 으로 바꿔 적을 필요가 없습니다.

> 자가채점이 `spec12 == saved['seller_intro']` 를 확인하는 이유: 프롬프트를 코드에 그대로 적어 두고 파일만 따로 만들어도 앞의 검사는 통과하기 때문입니다. **파일에서 읽어 온 것**이라야 이 문제를 푼 것입니다.

---
수고했어요! **모델·메시지(이미지 포함)·프롬프트 템플릿·출력 파서·체인(invoke·batch·stream)** 의 기본기를 익혔습니다. LV2 에서는 체인을 잇고, 함수를 부품으로 만들고, 대화 기록을 다룹니다.